# Maritime Atlas - подключение к своему GitHub

Ноутбук скачивает публичные GeoJSON из собственного репозитория и открывает веб-карту на GitHub Pages. **До запуска измените `GITHUB_USER`.** Без запуска локального приёмника AIS веб-версия отображает только статический срез.

In [ ]:
from pathlib import Path
import json
import requests
import pandas as pd
from IPython.display import display, IFrame

In [ ]:
GITHUB_USER = "YOUR_GITHUB_USERNAME"
REPO = "maritime-atlas"
BRANCH = "main"
assert GITHUB_USER != "YOUR_GITHUB_USERNAME", "Замените GITHUB_USER на ваш GitHub-логин"
RAW = f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO}/{BRANCH}/docs/data"
PAGES = f"https://{GITHUB_USER}.github.io/{REPO}/"

## 1. Загрузка трёх GeoJSON

Для публичного репозитория авторизация не нужна. При изменении данных на GitHub повторите ячейку. Слои могут быть пустыми до импорта AIS и статистики.

In [ ]:
def download_geojson(layer):
    response = requests.get(f"{RAW}/{layer}.geojson", timeout=30)
    response.raise_for_status()
    return response.json()

layers = {name: download_geojson(name) for name in ("density", "flows", "vessels")}
{name: len(value.get("features", [])) for name, value in layers.items()}

## 2. Таблица маршрутов и подтверждённые объёмы

In [ ]:
flows_df = pd.json_normalize([feature["properties"] for feature in layers["flows"].get("features", [])])
columns = [name for name in ("route_id", "name_ru", "cargo_ru", "cargo_tonnes", "year", "status", "source_url") if name in flows_df.columns]
display(flows_df[columns])

## 3. Интерактивная карта GitHub Pages

In [ ]:
display(IFrame(src=PAGES, width="100%", height=700))

## 4. Сохранение GeoJSON локально

In [ ]:
output_dir = Path("maritime_layers")
output_dir.mkdir(exist_ok=True)
for layer, geojson in layers.items():
    (output_dir / f"{layer}.geojson").write_text(json.dumps(geojson, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Сохранено в {output_dir.resolve()}")

## 5. Установка Python-пакета из собственного репозитория (необязательно)

Если планируете запускать обработку данных прямо из ноутбука, выполните отдельной ячейкой:

```python
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", f"git+https://github.com/{GITHUB_USER}/{REPO}.git"])
from maritime_atlas.processing.density import make_density
```

Для локальных AISStream-данных потребуется `.env` на MacBook. Не размещайте ключ в коде ноутбука, предназначенного для публичного GitHub.